# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [38]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [49]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.environ.get('DEEPSEEK_API_KEY')

if api_key and api_key.startswith('sk-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'deepseek-flash'
openai = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

API key looks good so far


In [10]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [11]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [13]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [50]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model='deepseek-flash',
        messages=[
            {"role": "system", "content":link_system_prompt},
            {"role": "user", "content":get_links_user_prompt(url)}
        ]
    )
    
    print(response.choices[0].message.content)
    print(type(response.choices[0].message.content))

In [47]:
select_relevant_links("https://edwarddonner.com")

```
{
  "links": [
    {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},
    {"type": "company page", "url": "https://edwarddonner.com/"},
    {"type": "careers/jobs page", "url": "https://edwarddonner.com/posts/"}
  ]
}
```

I excluded the following links as they do not align with the purpose of a company brochure:

- News article link: https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
- Social media links (excluding the linkedin profile)
- Blogs with duplicate entries (e.g., duplicate posts with the same title and content)
- Terms of Service link
- Email link
- Links to courses (e.g., https://edwarddonner.com/curriculum/, https://edwarddonner.com/avatar/, etc.)
- Links to games (e.g., https://edwarddonner.com/connect-four/, https://edwarddonner.com/outsmart/)
- Duplicate links
- Relative links (#wp--skip-link--target)
<class 'str'>


In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [53]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling deepseek-flash
Found 8 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [54]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-flash
Found 14 relevant links


{'links': [{'type': 'company page',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'support page', 'url': 'https://huggingface.co/support'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'community discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [67]:
def fetch_page_and_all_relevant_links(url):
    content = fetch_website_contents(url)
    relevant_link = select_relevant_links(url)
    result = f"###Landing Page\n\n {content} \n\n ###Relevant Link"
    for link in relevant_link["links"]:
        result += link["type"]
        result += fetch_website_contents(link["url"])
    return result

In [68]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling deepseek-flash
Found 11 relevant links
###Landing Page

 Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
convaiinnovations/laya
Updated
1 day ago
•
3.54k
Qwen/Qwen-Image-2.1
Updated
4 days ago
•
42.5k
•
2.26k
abenzerps/Qwen-Image-2.1-Uncensored-GGUF
Updated
2 days ago
•
716k
•
1.71k
prism-ml/Ternary-Bonsai-2-27B-gguf
Updated
about 14 hours ago
•
3.11M
•
2.06k
XingChen-AGI/Xing4.0-29B

In [76]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [70]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [71]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-flash
Found 18 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n###Landing Page\n\n Hugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nconvaiinnovations/laya\nUpdated\n1 day ago\n•\n3.54k\nQwen/Qwen-Image-2.1\nUpdated\n4 days ago\n•\n42.5k\n•\n2.26k\n

In [72]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [73]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-flash
Found 10 relevant links


# Hugging Face – The AI community building the future

## About
Hugging Face is the collaboration platform for the machine learning community. The Hugging Face Hub is a central place where anyone can discover, create, and collaborate on models, datasets, and applications.

## What we do
- **Models:** Browse 2M+ models for a wide range of AI tasks.
- **Datasets:** Explore 500k+ datasets for training, evaluation, and research.
- **Spaces:** Run and share 1M+ machine learning applications.
- **Buckets:** A new storage option for ML workflows.
- **Enterprise:** Team & Enterprise, Hugging Face PRO, Enterprise Support, Inference Providers, Inference Endpoints, and Storage Buckets.
- **Community & learning:** Docs, Tasks, HuggingChat, Collections, Languages, Organizations, Blog, Daily Papers, Hardware, Learn, Discord, Forum, and GitHub.

## Culture
- **Community-driven:** Built around open collaboration and the belief that AI should be built together.
- **Open source at heart:** Active thought leadership on open-source AI ecosystems, from DeepSeek to global compute landscapes.
- **Knowledge sharing:** Daily papers, blog posts, learning tracks, forums, and Discord keep the community informed.
- **Fast-moving and experimental:** Trending models, demos, and applications are updated daily.
- **Inclusive:** A central hub for researchers, developers, hobbyists, startups, and enterprises to contribute.

## Customers and who we serve
- **Machine learning community:** Researchers, developers, data scientists, and students.
- **Enterprises and teams:** Organizations needing inference endpoints, enterprise support, and storage.
- **Individual builders:** Users who host public models, datasets, and apps, or upgrade to PRO.
- **Open-source organizations and AI startups:** Groups collaborating on the Hub.

## Careers and recruiting
The provided pages do not list specific open roles. For recruits, Hugging Face offers a mission-driven, community-oriented environment focused on open machine learning. To explore opportunities, follow the company page, read the blog, join the Discord or Forum, and engage on GitHub.

## By the numbers
- 2M+ models
- 500k+ datasets
- 1M+ applications
- 106,808 followers on the company page

## Brand
- Colors: #FFD21E, #FF9D00, #6B7280
- Bio: “Hugging Face is the collaboration platform for the machine learning community. The Hugging Face Hub works as a central place where anyone...”

## Get started
Explore AI apps, browse models and datasets, or sign up at [huggingface.co](https://huggingface.co).

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [74]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [75]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-flash
Found 14 relevant links


# Hugging Face — Company Brochure

**The AI community building the future.**

Hugging Face is the collaboration platform for the machine learning community. Its Hub is a central place where anyone can collaborate on models, datasets, and applications.

## At a Glance
- 2M+ models
- 500k+ datasets
- 1M+ applications
- 106,808 followers on the company page
- Open, community-driven machine learning platform
- Enterprise, inference, and storage offerings

## What Hugging Face Offers
- **Models** — browse, share, and discover 2M+ models.
- **Datasets** — access and contribute to 500k+ datasets.
- **Spaces** — run and share ML applications; 1M+ apps.
- **Buckets** — new storage offering.
- **HuggingChat, Tasks, Collections, Languages, Organizations** — tools for discovery and collaboration.
- **Docs, Hardware, Learn** — resources for building and learning.
- **Community** — Blog, Posts, Daily Papers, Discord, Forum, GitHub.

## Solutions for Teams & Enterprises
Hugging Face serves individual builders and organizations through:
- Team & Enterprise
- Hugging Face PRO
- Enterprise Support
- Inference Providers
- Inference Endpoints
- Storage Buckets

These solutions indicate a commercial platform for teams that need collaboration, inference, support, and storage.

## Culture & Community
Hugging Face’s culture is built around open collaboration and community:
- “The AI community building the future.”
- A central place where anyone can collaborate on ML.
- Public hosting and sharing of models, datasets, and applications.
- Active channels for discussion and contribution: Discord, Forum, GitHub, Blog, and Papers.
- “Create, discover and collaborate on ML better.”

## Customers & Users
- ML researchers, developers, data scientists, and AI builders.
- Teams and enterprises using Hugging Face’s enterprise, inference, and storage products.
- The global open-source ML community.
- Organizations that want to publish, discover, and deploy machine learning assets.

## Careers & Recruiting
The provided pages do not include a careers page, open job listings, or hiring details. Prospective recruits should check the Hugging Face company page, GitHub, Discord, and blog for opportunities and ways to contribute.

## For Investors
The provided pages do not include financial, funding, or valuation information. They do show scale and commercial surface area:
- 2M+ models, 500k+ datasets, and 1M+ applications.
- Enterprise offerings such as Team & Enterprise, Enterprise Support, Inference Providers, Inference Endpoints, and Storage Buckets.
- A large, active community with 106,808 followers and ongoing papers/articles.

## Brand & Connect
- Brand colors: #FFD21E, #FF9D00, #6B7280.
- Official logos available in .svg, .png, and .ai formats.
- Explore AI Apps, browse 2M+ models, sign up, or follow the company page.
- Main site: huggingface.co.

In [77]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling deepseek-flash
Found 15 relevant links


# 🤗 Hugging Face
### The AI community building the future (and, apparently, hoarding 2 million models)

---

## What is this place?

Imagine a library. Now imagine the library has **2,000,000+ models**, **1,000,000+ applications**, and **500,000+ datasets**, and instead of a stern librarian shushing you, there's a Discord server where someone is genuinely excited to help you fine-tune a 27-billion-parameter monstrosity at 2 a.m.

That's Hugging Face. The platform where the machine learning community collaborates on models, datasets, and applications. The tagline is *"The AI community building the future."* The reality is a bit more chaotic and a lot more fun.

We are, officially, **The Home of Machine Learning**. Unofficially, we're the place where your GPU goes to have a personality crisis.

---

## 🧠 What actually happens here

- **Models** — Browse 2M+ of them. Some generate images. Some generate video with synchronized soundtracks. Some do things we're not entirely sure are legal in all jurisdictions (looking at you, "Uncensored" anything).
- **Spaces** — 1M+ running applications. Animate a picture into a short video. Generate an image from text. Watch a community benchmark evolve in real time while you eat lunch.
- **Datasets** — 500k+ of them, including `wikimedia/wikipedia`, a complete arXiv dump, and a whole lot of things with "Filtered" in the name.
- **Buckets** — new! Storage. Because even AI needs somewhere to put its stuff.
- **HuggingChat, Daily Papers, Blog, Collections, Languages** — the community caffeine drip, administered intravenously.

Everything is public, and everything is collaborative. This is the "move fast" part of the brochure.

---

## 🏢 For Teams & Enterprise

Yes, we have a serious side. It wears a slightly nicer hoodie.

- **Enterprise Support** — humans, on purpose, when you need them
- **Inference Endpoints** — deploy models without becoming a Kubernetes monk
- **Inference Providers** — plug into the ecosystem
- **Storage Buckets** — carry your weights without emotional baggage
- **Hugging Face PRO** — for the individual who wants more and is willing to admit it

Whether you're a solo tinkerer or a Fortune 500 that just discovered the word "transformer," there's a tier with your name on it.

---

## 🎭 Culture (as observed in the wild)

The homepage reads like a live feed of human curiosity. Trending this week: a ternary-quantized 27B bonsai, a video model with a soundtrack, a decision-index benchmark, and something called "*Laya*" that 3,540 people found in one day.

That's the culture: **open, fast, slightly unhinged, deeply collaborative.**

- 🧑‍🤝‍🧑 Community-first — Discord, Forum, GitHub, all first-class citizens
- 📖 Radically open — public models, public datasets, public papers, public opinions
- 🚀 Ship-it energy — "Updated 1 day ago" is not a brag, it's a lifestyle
- 🧪 Everyone's a researcher — the next breakthrough might come from a hobbyist with a used 3090

---

## 👥 Customers & Community

- **Researchers** who need a dataset that definitely exists somewhere
- **Startups** who need 2M models and zero procurement meetings
- **Enterprises** who need SLAs, security, and a support engineer named Kevin
- **Students** who need to finish a thesis by Thursday
- **Hobbyists** who need to know whether a bonsai can be ternary

They all show up in the same place. That's the magic trick.

---

## 💼 Careers

We didn't get the careers page in this batch of reading material, which is honestly very on-brand for a company that moves this fast.

But based on the evidence: if you like **open source**, **shipping things**, **helping strangers debug their training loops**, and **explaining to your relatives that no, you don't "make the chatbots,"** you'd probably fit in.

Check the actual careers page for openings. We assume there are many. There usually are.

---

## 🏁 The Pitch

**To customers:** Everything you need to build with AI, already here, already free to browse.

**To investors:** 2M+ models. 1M+ apps. 500k+ datasets. The community didn't just adopt us — it *built* us.

**To recruits:** Come help build the future. The code's public, the Discord's lively, and someone is definitely already reviewing your pull request.

---

*🤗 Hugging Face — The AI community building the future.*
*Come for the models. Stay for the arguments about quantization.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>